# Day 4 · 07 · Vectorisation

A vectorised operation works on a **whole column at once**. The loop still happens, but in compiled code inside pandas and NumPy instead of in Python, one row at a time.

**In this notebook**
- Timing a row loop, `apply` and a vectorised expression
- Conditional logic: `np.where`, `np.select`, `pd.cut`
- Lookups with `map`
- Text and dates: the `.str` and `.dt` accessors
- Built-in aggregations versus Python functions
- Boolean reductions and broadcasting
- What to do when logic cannot be vectorised

**Dataset:** the retail CSV files. Order lines are resampled to 20,000 rows so that timings are measurable.

In [1]:
import re
import timeit
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")

lines = items.merge(orders, on="order_id")
big = lines.sample(20_000, replace=True, random_state=42).reset_index(drop=True)


def best_ms(function, repeat=3):
    """Best of `repeat` runs, in milliseconds."""
    return min(timeit.repeat(function, number=1, repeat=repeat)) * 1000


print(big.shape)
big.head(3)

(20000, 11)


,order_id,line_no,product_id,quantity,unit_price,discount,customer_id,order_date,status,channel,staff_id
0,10375,1,P011,2,27.0,0.2,C0181,2024-05-14,completed,web,NaN
1,11711,2,P023,2,129.0,0.1,C0147,2025-04-26,completed,web,NaN
2,11391,3,P025,1,32.0,0.0,C0221,2025-02-07,completed,store,S10


## 1. Six ways to compute one column
The task: `revenue = quantity × unit_price × (1 − discount)` for every row.

| Approach | How it works |
| --- | --- |
| `iterrows` | builds a Series for every row, then runs Python arithmetic |
| `apply(axis=1)` | calls a Python function once per row |
| `itertuples` | yields a lightweight tuple per row |
| `zip` over columns | plain Python loop over the values |
| pandas expression | whole-column arithmetic in compiled code |
| NumPy arrays | the same, without index alignment |

In [2]:
def with_iterrows():
    result = []
    for _, row in big.iterrows():
        result.append(row["quantity"] * row["unit_price"] * (1 - row["discount"]))
    return result


def with_apply():
    return big.apply(lambda row: row["quantity"] * row["unit_price"] * (1 - row["discount"]), axis=1)


def with_itertuples():
    return [row.quantity * row.unit_price * (1 - row.discount) for row in big.itertuples(index=False)]


def with_zip():
    return [q * p * (1 - d) for q, p, d in zip(big["quantity"], big["unit_price"], big["discount"])]


def vectorised():
    return big["quantity"] * big["unit_price"] * (1 - big["discount"])


def with_numpy():
    return big["quantity"].to_numpy() * big["unit_price"].to_numpy() * (1 - big["discount"].to_numpy())


print("all approaches agree:", all(np.allclose(f(), vectorised()) for f in [with_iterrows, with_apply, with_itertuples, with_zip, with_numpy]))

all approaches agree: True


In [3]:
timings = {
    "iterrows loop": best_ms(with_iterrows, repeat=1),
    "apply(axis=1)": best_ms(with_apply, repeat=1),
    "itertuples": best_ms(with_itertuples),
    "zip over columns": best_ms(with_zip),
    "pandas expression": best_ms(vectorised, repeat=7),
    "NumPy arrays": best_ms(with_numpy, repeat=7),
}

result = pd.DataFrame({"milliseconds": timings})
result["times_slower_than_pandas_expression"] = result["milliseconds"] / result.loc["pandas expression", "milliseconds"]
result.round(2)

,milliseconds,times_slower_than_pandas_expression
iterrows loop,500.89,3743.59
apply(axis=1),156.81,1171.96
itertuples,100.77,753.12
zip over columns,5.03,37.58
pandas expression,0.13,1.00
NumPy arrays,0.07,0.49


The exact numbers depend on the machine, but the ranking is stable. Every row-by-row approach is slower than the whole-column expression: `iterrows` and `apply(axis=1)` by the widest margin, a plain `zip` loop over the columns by the smallest.

The vectorised version is also the shortest and the easiest to read.

In [4]:
big["revenue"] = big["quantity"] * big["unit_price"] * (1 - big["discount"])

## 2. Conditional logic without `if`
An `if` statement handles one value. For a column use:

| Need | Tool |
| --- | --- |
| two outcomes | `np.where(condition, value_if_true, value_if_false)` |
| several conditions, first match wins | `np.select([conditions], [choices], default=...)` |
| numeric ranges | `pd.cut` |
| clamp to limits | `.clip(lower, upper)` |

In [5]:
big["size"] = np.where(big["revenue"] >= 100, "large", "small")

conditions = [
    big["status"] == "cancelled",
    big["revenue"] >= 250,
    big["revenue"] >= 100,
]
choices = ["none", "free", "reduced"]
big["shipping"] = np.select(conditions, choices, default="standard")

big["shipping"].value_counts()

shipping
standard    13165
reduced      4141
none         1548
free         1146
Name: count, dtype: int64

In [6]:
# The same rule as a Python function applied row by row.
def shipping_rule(row):
    if row["status"] == "cancelled":
        return "none"
    if row["revenue"] >= 250:
        return "free"
    if row["revenue"] >= 100:
        return "reduced"
    return "standard"


def select_version():
    return np.select(conditions, choices, default="standard")


print("same result:", (big.apply(shipping_rule, axis=1) == big["shipping"]).all())
print(f"apply(axis=1): {best_ms(lambda: big.apply(shipping_rule, axis=1), repeat=1):8.2f} ms")
print(f"np.select    : {best_ms(select_version):8.2f} ms")

same result: True
apply(axis=1):   170.45 ms
np.select    :     0.24 ms


In [7]:
# Ranges with pd.cut, limits with clip.
big["value_band"] = pd.cut(big["revenue"], bins=[0, 25, 100, 250, np.inf], labels=["under 25", "25-100", "100-250", "250+"])
big["capped_quantity"] = big["quantity"].clip(upper=3)

print(big["value_band"].value_counts().sort_index().to_string(), "\n")
print("quantity max:", big["quantity"].max(), "| after clip(upper=3):", big["capped_quantity"].max())

value_band
under 25     2969
25-100      11367
100-250      4424
250+         1240 

quantity max: 5 | after clip(upper=3): 3


## 3. Lookups with `map`
To translate values through a table, use `Series.map` with a dictionary or with a Series indexed by the key. It is a vectorised lookup, and for a single column it is simpler than a merge.

In [8]:
unit_cost = products.set_index("product_id")["unit_cost"]
category = products.set_index("product_id")["category"]

big["category"] = big["product_id"].map(category)
big["profit"] = big["quantity"] * (big["unit_price"] * (1 - big["discount"]) - big["product_id"].map(unit_cost))
big["sales_type"] = big["channel"].map({"web": "online", "mobile": "online", "store": "in person"})

big.groupby(["sales_type", "category"])[["revenue", "profit"]].sum().round(0).head(6)

revenue   profit
sales_type category                      
in person  Books         82883.0  28785.0
           Electronics  164832.0  61680.0
           Home          49688.0  20922.0
           Sports        79555.0  32157.0
           Toys          32889.0   9647.0
online     Books        259816.0  91340.0

## 4. Text: the `.str` accessor
`.str` applies a string method to every value and passes missing values through untouched. Methods can be chained, and `.str[n]` indexes into the result of a split.

The speed gain over a Python loop is usually much smaller for text than for numbers, because the work is still done string by string. The main benefits are clarity and correct handling of missing values. Practice exercise 4 lets you measure it.

In [9]:
emails = customers["email"]

pd.DataFrame(
    {
        "email": emails,
        "user": emails.str.split("@").str[0],
        "domain": emails.str.split("@").str[1],
        "digits": emails.str.extract(r"(\d+)@")[0],
        "first_name": customers["name"].str.split().str[0].str.upper(),
        "name_length": customers["name"].str.len(),
    }
).head(4)

,email,user,domain,digits,first_name,name_length
0,rohan.lopez1@example.com,rohan.lopez1,example.com,1,ROHAN,11
1,rohan.fischer2@example.com,rohan.fischer2,example.com,2,ROHAN,13
2,felix.garcia3@example.com,felix.garcia3,example.com,3,FELIX,12
3,lucia.murphy4@example.com,lucia.murphy4,example.com,4,LUCIA,12


## 5. Dates: the `.dt` accessor
Every date component and many date calculations are available for a whole column through `.dt`.

In [10]:
dates = big["order_date"]

pd.DataFrame(
    {
        "order_date": dates,
        "month": dates.dt.to_period("M"),
        "weekday": dates.dt.day_name(),
        "is_weekend": dates.dt.day_of_week >= 5,
        "days_to_month_end": (dates + pd.offsets.MonthEnd(0) - dates).dt.days,
    }
).head(4)

,order_date,month,weekday,is_weekend,days_to_month_end
0,2024-05-14,2024-05,Tuesday,False,17
1,2025-04-26,2025-04,Saturday,True,4
2,2025-02-07,2025-02,Friday,False,21
3,2024-03-15,2024-03,Friday,False,16


## 6. Aggregations: use the built-in names
`groupby(...).sum()` or `.agg("sum")` runs in compiled code. Passing a Python function, such as a `lambda`, makes pandas call it once **per group**.

In [11]:
def builtin_sum():
    return big.groupby("customer_id")["revenue"].sum()


def lambda_sum():
    return big.groupby("customer_id")["revenue"].agg(lambda values: values.sum())


print("same result:", np.allclose(builtin_sum(), lambda_sum()))
print(f"built-in 'sum'   : {best_ms(builtin_sum):7.2f} ms")
print(f"lambda per group : {best_ms(lambda_sum):7.2f} ms  ({big['customer_id'].nunique()} groups)")

same result: True
built-in 'sum'   :    1.36 ms
lambda per group :    4.25 ms  (205 groups)


## 7. Boolean reductions
A comparison gives a boolean column. Reduce it directly instead of looping and counting:
- `.sum()` counts the `True` values
- `.mean()` gives their share
- `.any()` / `.all()` answer "is there one?" / "are they all?"

In [12]:
discounted = big["discount"] > 0

print("discounted lines     :", discounted.sum())
print("share discounted     :", round(discounted.mean(), 3))
print("any discount over 20%:", (big["discount"] > 0.20).any())
print("all quantities >= 1  :", (big["quantity"] >= 1).all())

discounted lines     : 8159
share discounted     : 0.408
any discount over 20%: False
all quantities >= 1  : True


## 8. Broadcasting
Operations between a table and a row of column statistics are aligned automatically. Standardising every numeric column needs no loop over columns or rows.

In [13]:
numeric = big[["quantity", "unit_price", "discount", "revenue"]]
standardised = (numeric - numeric.mean()) / numeric.std()

standardised.agg(["mean", "std"]).round(3)

,quantity,unit_price,discount,revenue
mean,0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0


In [14]:
# Running totals and comparisons with the previous row are vectorised too.
ordered = big.sort_values(["customer_id", "order_date"])
ordered["customer_running_revenue"] = ordered.groupby("customer_id")["revenue"].cumsum()

ordered[["customer_id", "order_date", "revenue", "customer_running_revenue"]].round({"revenue": 2, "customer_running_revenue": 2}).head(4)

,customer_id,order_date,revenue,customer_running_revenue
4989,C0001,2024-02-03,12.0,12.0
5227,C0001,2024-02-03,12.0,24.0
5532,C0001,2024-02-03,50.0,74.0
7804,C0001,2024-02-03,50.0,124.0


## 9. When you really need a Python function
Some logic only exists as a Python function: a call to a library, a parser, a complicated rule. Then:
- Loop over the **values of the columns you need** with a list comprehension, or use `Series.map(function)`.
- Use `itertuples` if you need several columns.
- Avoid `iterrows` and `apply(axis=1)`.
- `np.vectorize` makes a function accept arrays, but it is still a Python loop.

In [15]:
def slug(text):
    """'Noise-Cancelling Headphones' -> 'noise-cancelling-headphones'"""
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


products["slug"] = [slug(name) for name in products["product_name"]]
products[["product_name", "slug"]].head(3)

,product_name,slug
0,Wireless Mouse,wireless-mouse
1,Mechanical Keyboard,mechanical-keyboard
2,USB-C Hub,usb-c-hub


## Loop to vectorised: a translation table
| Loop idea | Vectorised form |
| --- | --- |
| `for row: a * b` | `df["a"] * df["b"]` |
| `if ... else ...` | `np.where` |
| `if / elif / else` | `np.select`, `pd.cut` |
| dictionary lookup per row | `series.map(mapping)` |
| string method per row | `series.str.method()` |
| date part per row | `series.dt.part` |
| count rows that match | `(condition).sum()` |
| running total | `cumsum`, grouped if needed |
| compare with previous row | `shift`, `diff` |

## Key takeaways
- Think in columns, not rows.
- Arithmetic, comparisons, `np.where`, `np.select`, `map`, `.str` and `.dt` cover most row logic.
- Use built-in aggregation names instead of lambdas.
- Measure before and after. Vectorise the slow part, not everything.

## Practice
1. Add a `discount_band` column: "none", "low" (up to 10%), "high" (above 10%), using `np.select`.
2. Compute each line's share of its order's revenue without a loop.
3. Replace the `shipping_rule` function with `pd.cut` plus one `np.where`.
4. Time `big["channel"].str.upper()` against a list comprehension. Which is faster on your machine?